In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.models import resnet18
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import accuracy_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from tqdm.auto import tqdm
import time
import json

# Where trained models and results are stored. On Colab they go to Google Drive, so the
# run can resume after the runtime disconnects (training all 210 models takes many hours).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/PDL_CA0'
except ImportError:
    SAVE_DIR = '.'
MODEL_DIR = os.path.join(SAVE_DIR, 'saved_models')
RESULT_DIR = os.path.join(SAVE_DIR, 'results')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('Saving to:', SAVE_DIR)

In [ ]:
# Step 1: Load and Preprocess CIFAR-10 Dataset

def load_cifar10_balanced_or_imbalanced(batch_size=128, n_elements_per_class=4000, class_to_augment=None, augmented_size=6000, other_size=3777, download=True, train_ratio=0.8, split_seed=0):
    # Normalization transform
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    ])
    
    # Load the full CIFAR-10 train dataset
    full_train_set = datasets.CIFAR10(root='./data', train=True, download=download, transform=transform)
    targets = np.array(full_train_set.targets)
    
    # Number of elements per class, i.e. the vector N = [N_1, ..., N_10]
    if class_to_augment is not None:
        N = [augmented_size if k == class_to_augment else other_size for k in range(10)]
    else:
        N = [n_elements_per_class] * 10
    
    # Split each class proportionally according to N. The split uses a fixed seed, so the
    # test set is the same every time the loader is called and never leaks into training.
    rng = np.random.default_rng(split_seed)
    train_indices, test_indices = [], []
    for k in range(10):
        class_indices = rng.permutation(np.where(targets == k)[0])[:N[k]]
        n_train = int(N[k] * train_ratio)
        train_indices.append(class_indices[:n_train])
        test_indices.append(class_indices[n_train:])
    train_indices = np.concatenate(train_indices)
    test_indices = np.concatenate(test_indices)

    # Create Subset objects for train and test sets
    train_subset = Subset(full_train_set, train_indices)
    test_subset = Subset(full_train_set, test_indices)
    
    # Create DataLoaders for the subsets
    # Colab provides 2 CPU cores
    num_workers = min(2, os.cpu_count() or 1)
    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    
    return train_loader, test_loader


def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=20, device=None):
    model.to(device)
    train_accuracies = []
    test_accuracies  = [] 

    # Mixed precision on the GPU makes each epoch considerably faster on Colab
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    
    
    for epoch in tqdm(range(num_epochs), desc='Training Progress'):
        model.train()
        epoch_start_time = time.time()
        all_targets_train = []
        all_predictions_train = []
        
        for data, target in train_loader:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                output = model(data)
                loss = criterion(output, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, predictions = torch.max(output, 1)
            all_targets_train.extend(target.cpu().numpy())
            all_predictions_train.extend(predictions.cpu().numpy())

        # Compute accuracy for the current epoch using sklearn's accuracy_score
        epoch_accuracy = 100 * accuracy_score(all_targets_train, all_predictions_train)
        train_accuracies.append(epoch_accuracy)

        # Evaluation after each epoch
        test_accuracy, per_class_accuracies, all_targets_test, all_predictions_test = evaluate_model(model, test_loader, device)
        test_accuracies.append(test_accuracy)

        epoch_duration = time.time() - epoch_start_time
        print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}, Accuracy: {epoch_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%, Duration: {epoch_duration:.2f}s')
        
    return train_accuracies, test_accuracies


def evaluate_model(model, test_loader, device):
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    # Calculate overall accuracy
    accuracy = 100 * accuracy_score(all_targets, all_predictions)

    # Per-class accuracy: the fraction of test samples of class k that are classified as k,
    # which is the per-class recall
    per_class_accuracies = recall_score(all_targets, all_predictions, labels=list(range(10)), average=None, zero_division=0)
    
    # Return overall accuracy, per-class accuracies, and the lists of all targets and predictions
    return accuracy, per_class_accuracies, all_targets, all_predictions

In [ ]:
# Step 2: Define ResNet18 Model
def build_model(seed):
    # A fresh ResNet18 with randomly initialized weights (no ImageNet pretraining) and a
    # 10-way output layer. The seed controls the random initialization of all layers.
    torch.manual_seed(seed)
    model = resnet18(weights=None, num_classes=10)
    return model.to(device)
            
# Step 3: Train and Evaluate the Model
num_epochs = 20
num_initializations = 10
n_elements_per_class = 4000

# Define loss function
criterion = nn.CrossEntropyLoss()

# The balanced dataset is the same for all initializations
train_loader, test_loader = load_cifar10_balanced_or_imbalanced(
    batch_size=128,
    n_elements_per_class=n_elements_per_class,
    download=True,
    train_ratio=0.8  # 80% of data for training
)

# Training and evaluation process
all_train_accuracies = []
all_test_accuracies = []
for init in range(num_initializations):
    print(f"Initialization {init + 1}/{num_initializations}")
    model_path = os.path.join(MODEL_DIR, f'resnet18_model_init_{init}.pt')

    # Skip models that were already trained before a disconnect
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location='cpu')
        all_train_accuracies.append(checkpoint['train_accuracies'])
        all_test_accuracies.append(checkpoint['test_accuracies'])
        print('  already trained, loaded from', model_path)
        continue

    model = build_model(seed=init)
    optimizer = optim.Adam(model.parameters())
    
    # Train and evaluate the model
    train_accuracies, test_accuracies = train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs, device=device)
    all_train_accuracies.append(train_accuracies)
    all_test_accuracies.append(test_accuracies)
    
    # Save the model together with its accuracy curves
    torch.save({
        'state_dict': model.state_dict(),
        'train_accuracies': train_accuracies,
        'test_accuracies': test_accuracies,
    }, model_path)

    # Free up memory
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

# Train and test accuracy averaged over the 10 random initializations
mean_train = np.mean(all_train_accuracies, axis=0)
std_train = np.std(all_train_accuracies, axis=0)
mean_test = np.mean(all_test_accuracies, axis=0)
std_test = np.std(all_test_accuracies, axis=0)
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, mean_train, label='Training Accuracy')
plt.fill_between(epochs, mean_train - std_train, mean_train + std_train, alpha=0.2)
plt.plot(epochs, mean_test, label='Test Accuracy')
plt.fill_between(epochs, mean_test - std_test, mean_test + std_test, alpha=0.2)
plt.title(f'Training and Test Accuracy Averaged over {num_initializations} Initializations')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.ylim([0, 100])
plt.legend()
plt.show()

In [ ]:
# Step 4: Investigate the per-class accuracy of the ten trained models
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

baseline_per_class_accuracies = []
for init in range(num_initializations):
    model = resnet18(weights=None, num_classes=10)
    checkpoint = torch.load(os.path.join(MODEL_DIR, f'resnet18_model_init_{init}.pt'), map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    model.to(device)
    _, per_class_accuracies, _, _ = evaluate_model(model, test_loader, device)
    baseline_per_class_accuracies.append(per_class_accuracies)
    del model
    torch.cuda.empty_cache()

baseline_per_class_accuracies = np.array(baseline_per_class_accuracies)
average_per_class_accuracies = baseline_per_class_accuracies.mean(axis=0)

# Plot the per-class accuracies
plt.figure(figsize=(12, 6))
plt.bar(classes, average_per_class_accuracies, yerr=baseline_per_class_accuracies.std(axis=0), capsize=4)
plt.title(f'Average Per-Class Accuracy Over {num_initializations} Models (Balanced Dataset)')
plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.show()

In [ ]:
# Step 5: Train on Imbalanced dataset, with (Step 7) and without class-weighted loss

# Storage for final analysis
all_overall_test_accuracies_unweighted = []
all_overall_test_accuracies_weighted = []
all_per_class_accuracies_unweighted = []
all_per_class_accuracies_weighted = []

for class_to_augment in range(10):  # Iterate over each class to augment
    # Load the imbalanced dataset loaders
    train_loader, test_loader = load_cifar10_balanced_or_imbalanced(
        batch_size=128,
        class_to_augment=class_to_augment,  # augmenting current class
        augmented_size=6000,
        other_size=3777,
        train_ratio=0.8  # 80% of data for training
    )
    
    # Compute class weights for the imbalanced dataset
    train_subset = train_loader.dataset
    targets = np.array(train_subset.dataset.targets)[train_subset.indices]
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(10),
        y=targets
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    
    # Initialize the loss functions
    criterion_unweighted = nn.CrossEntropyLoss()
    criterion_weighted = nn.CrossEntropyLoss(weight=class_weights)

    # Lists to hold accuracies for each initialization
    init_test_accuracies_unweighted = []
    init_test_accuracies_weighted = []
    init_per_class_accuracies_unweighted = []
    init_per_class_accuracies_weighted = []
    for init in range(num_initializations):  # Train ten models for this dataset
        print(f"Class {class_to_augment}, Initialization {init + 1}/{num_initializations}")
        result_path = os.path.join(RESULT_DIR, f'imbalanced_class{class_to_augment}_init{init}.json')

        # Skip runs that were already finished before a disconnect
        if os.path.exists(result_path):
            with open(result_path) as f:
                result = json.load(f)
            init_test_accuracies_unweighted.append(result['test_accuracy_unweighted'])
            init_test_accuracies_weighted.append(result['test_accuracy_weighted'])
            init_per_class_accuracies_unweighted.append(np.array(result['per_class_unweighted']))
            init_per_class_accuracies_weighted.append(np.array(result['per_class_weighted']))
            print('  already trained, loaded from', result_path)
            continue
        
        # A new randomly initialized model. The weighted run below uses the same seed, so
        # the two losses are compared from the same starting weights.
        model = build_model(seed=init)
        optimizer = optim.Adam(model.parameters())

        # Train with unweighted loss
        train_accuracies_unweighted, test_accuracies_unweighted = train_model(
            model, train_loader, test_loader, criterion_unweighted, optimizer, num_epochs, device=device
        )
        test_accuracy_unweighted, per_class_accuracy_unweighted, _, _ = evaluate_model(model, test_loader, device)
        init_test_accuracies_unweighted.append(test_accuracy_unweighted)
        init_per_class_accuracies_unweighted.append(per_class_accuracy_unweighted)
        del model, optimizer

        # A new randomly initialized model with the same seed
        model = build_model(seed=init)
        optimizer = optim.Adam(model.parameters())

        # Train with weighted loss
        train_accuracies_weighted, test_accuracies_weighted = train_model(
            model, train_loader, test_loader, criterion_weighted, optimizer, num_epochs, device=device
        )
        test_accuracy_weighted, per_class_accuracy_weighted, _, _ = evaluate_model(model, test_loader, device)
        init_test_accuracies_weighted.append(test_accuracy_weighted)
        init_per_class_accuracies_weighted.append(per_class_accuracy_weighted)
        del model, optimizer

        # Do not save the models: only keep the statistics and free the memory
        with open(result_path, 'w') as f:
            json.dump({
                'test_accuracy_unweighted': test_accuracy_unweighted,
                'test_accuracy_weighted': test_accuracy_weighted,
                'per_class_unweighted': per_class_accuracy_unweighted.tolist(),
                'per_class_weighted': per_class_accuracy_weighted.tolist(),
                'train_curve_unweighted': train_accuracies_unweighted,
                'train_curve_weighted': train_accuracies_weighted,
                'test_curve_unweighted': test_accuracies_unweighted,
                'test_curve_weighted': test_accuracies_weighted,
            }, f)
        gc.collect()
        torch.cuda.empty_cache()

        # Set up the plot for training and testing accuracy side by side
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot the training accuracy curve for unweighted and weighted loss
        axes[0].plot(train_accuracies_unweighted, label='Unweighted', color='blue')
        axes[0].plot(train_accuracies_weighted, label='Weighted', color='green')
        axes[0].set_title('Training Accuracy')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        
        # Plot the test accuracy curve for unweighted and weighted loss
        axes[1].plot(test_accuracies_unweighted, label='Unweighted', color='red')
        axes[1].plot(test_accuracies_weighted, label='Weighted', color='orange')
        axes[1].set_title('Test Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()

        # Adjust the layout so that plots don't overlap with each other
        plt.tight_layout()
        plt.suptitle(f'Accuracy Curves for Class {class_to_augment}, Initialization {init + 1}')
        # This moves the title up slightly to avoid overlap with the subplots
        plt.subplots_adjust(top=0.85)
        plt.show()
    
    # After all initializations, average the accuracies
    avg_test_accuracy_unweighted = np.mean(init_test_accuracies_unweighted)
    avg_test_accuracy_weighted = np.mean(init_test_accuracies_weighted)
    avg_per_class_accuracies_unweighted = np.mean(init_per_class_accuracies_unweighted, axis=0)
    avg_per_class_accuracies_weighted = np.mean(init_per_class_accuracies_weighted, axis=0)

    # Store for final analysis
    all_overall_test_accuracies_unweighted.append(avg_test_accuracy_unweighted)
    all_overall_test_accuracies_weighted.append(avg_test_accuracy_weighted)
    all_per_class_accuracies_unweighted.append(avg_per_class_accuracies_unweighted)
    all_per_class_accuracies_weighted.append(avg_per_class_accuracies_weighted)

    # Plot per-class accuracies for unweighted vs weighted for the current class
    x = np.arange(10)  # the label locations
    width = 0.35  # the width of the bars

    fig, ax = plt.subplots()
    rects1 = ax.bar(x - width/2, avg_per_class_accuracies_unweighted, width, label='Unweighted')
    rects2 = ax.bar(x + width/2, avg_per_class_accuracies_weighted, width, label='Weighted')

    # Add some text for labels, title, and custom x-axis tick labels, etc.
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Per-Class Accuracy for Class {class_to_augment} (Unweighted vs Weighted)')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Class {i}' for i in range(10)])
    ax.legend()

    fig.tight_layout()
    plt.show()

In [ ]:
# Step 6: Illustrate the effect of data heterogeneity
# Convert the lists of per-class accuracies to NumPy arrays for easier manipulation
unweighted_accuracies_array = np.array(all_per_class_accuracies_unweighted)
weighted_accuracies_array = np.array(all_per_class_accuracies_weighted)

# Set up the matplotlib figure
fig, axes = plt.subplots(1, 2, figsize=(24, 10), sharey=True)

# Heatmap for unweighted accuracies
sns.heatmap(unweighted_accuracies_array, annot=True, cmap='viridis', fmt='.2f',
            ax=axes[0], cbar=True, xticklabels=[f'Class {i}' for i in range(10)],
            yticklabels=[f'Class {i}' for i in range(10)])
axes[0].set_title('Heatmap of Average Per-Class Accuracies (Unweighted)')
axes[0].set_xlabel('True Class')
axes[0].set_ylabel('Augmented Class')

# Heatmap for weighted accuracies
sns.heatmap(weighted_accuracies_array, annot=True, cmap='viridis', fmt='.2f',
            ax=axes[1], cbar=True, xticklabels=[f'Class {i}' for i in range(10)],
            yticklabels=[])
axes[1].set_title('Heatmap of Average Per-Class Accuracies (Weighted)')
axes[1].set_xlabel('True Class')
# axes[1].set_ylabel('Augmented Class')  # Not needed as it shares the y-axis with the first plot

plt.suptitle('Comparison of Per-Class Accuracies for Weighted vs Unweighted Loss', size=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust the padding to leave space for the suptitle
plt.show()

# Change in per-class accuracy relative to the balanced baseline from Step 4
fig, axes = plt.subplots(1, 2, figsize=(24, 10), sharey=True)
for ax, acc, name in [(axes[0], unweighted_accuracies_array, 'Unweighted'),
                      (axes[1], weighted_accuracies_array, 'Weighted')]:
    sns.heatmap(acc - average_per_class_accuracies, annot=True, cmap='RdBu', center=0, fmt='+.2f',
                ax=ax, cbar=True, xticklabels=[f'Class {i}' for i in range(10)],
                yticklabels=[f'Class {i}' for i in range(10)])
    ax.set_title(f'Per-Class Accuracy minus Balanced Baseline ({name})')
    ax.set_xlabel('True Class')
axes[0].set_ylabel('Augmented Class')
plt.tight_layout()
plt.show()